In [0]:
%pip install yfinance

import yfinance as yf
import pandas as pd
from datetime import datetime, timezone

In [0]:
# Define parameters via widgets
from datetime import datetime, timezone
import pandas as pd
import yfinance as yf

default_date = "2026-08-28"  # Last Friday: 28 August 2026
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "2. Target Schema")
dbutils.widgets.text("volume", "market_radar_landing", "3. Landing Volume")
dbutils.widgets.text("tickers", "AAPL,NVDA,MSFT,AMZN,TSLA,QQQ", "4. Tickers")
dbutils.widgets.text("target_date", default_date, "5. Target Date (YYYY-MM-DD)")
dbutils.widgets.text("landing_override_path", "", "6. Override Landing Path")

# Retrieve widget values
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
tickers = [t.strip() for t in dbutils.widgets.get("tickers").split(",")]
target_date = dbutils.widgets.get("target_date").strip()
override_path = dbutils.widgets.get("landing_override_path").strip()

# Resolve landing path
if override_path:
    landing_path = override_path
else:
    try:
        user_name = spark.sql("SELECT current_user()").collect()[0][0]
        if catalog == "workspace" or "gmail" in user_name:
            landing_path = f"/Workspace/Users/{user_name}/nasdaq_landing/landing/nasdaq_price"
        else:
            landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/nasdaq_price"
    except Exception:
        landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/nasdaq_price"

dbutils.fs.mkdirs(landing_path)
print(f"Target Landing Path: {landing_path}")


In [0]:
# Download 5-minute interval price bars for target date + recent window
next_date = (pd.to_datetime(target_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
records = []

print(f"Fetching 5-minute price ticks for date: {target_date} (and recent 5-day window for news coverage)...")
for symbol in tickers:
    try:
        # 1. Fetch target date 5m bars
        df_target = yf.download(symbol, start=target_date, end=next_date, interval="5m", progress=False)
        
        # 2. Also fetch recent 5-day 5m bars so that real news published in the last 5 days has price ticks
        df_recent = yf.download(symbol, period="5d", interval="5m", progress=False)
        
        frames = [f for f in [df_target, df_recent] if not f.empty]
        if frames:
            df_symbol = pd.concat(frames)
            if isinstance(df_symbol.columns, pd.MultiIndex):
                df_symbol.columns = df_symbol.columns.get_level_values(0)
            
            df_symbol = df_symbol.reset_index().drop_duplicates()
            # Standardize timestamp column name
            first_col = df_symbol.columns[0]
            if first_col.lower() in ["datetime", "date", "index"]:
                df_symbol.rename(columns={first_col: "PriceTimestamp"}, inplace=True)
            df_symbol["Symbol"] = symbol
            records.append(df_symbol)
            print(f"  -> {symbol}: fetched {len(df_symbol)} 5m bars")
        else:
            print(f"  -> {symbol}: no 5m data returned for {target_date}")
    except Exception as e:
        print(f"  -> Error fetching {symbol}: {e}")

if records:
    combined_df = pd.concat(records, ignore_index=True).drop_duplicates(subset=["Symbol", "PriceTimestamp"])
    output_file = f"{landing_path}/nasdaq_price_5m_{target_date}.csv"
    combined_df.to_csv(output_file, index=False)
    print(f"Successfully landed 5m price history ({len(combined_df)} rows) to: {output_file}")
else:
    print(f"No price records landed for {target_date}.")


In [0]:
# catalog = dbutils.widgets.get("catalog")
# schema = dbutils.widgets.get("schema")
# volume = dbutils.widgets.get("volume")
# landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/nasdaq_price"

# print("--- Volume File Listing ---")
# display(dbutils.fs.ls(landing_path))

# print("--- Preview Landed CSV ---")
# df_preview = spark.read.option("header", "true").csv(f"{landing_path}/nasdaq_price_history.csv")
# print(f"Total Landed Price Rows: {df_preview.count()}")
# display(df_preview.limit(10))